In [2]:
from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title
import torch
import chromadb
from langchain_core.messages import HumanMessage
from langchain_core.documents import Document
import ollama
from typing import List



/home/stark/Computer_Science/Artificial_Intelligence/machine_learning/AI_PDF_Reader/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def partition_document(file_path: str):
    """Extract elements from PDF using unstructured"""
    print(f"📄 Partitioning document: {file_path}")
    
    elements = partition_pdf(
        filename=file_path,  # Path to your PDF file
        strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
        infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
        extract_image_block_types=["Image"], # Grab images found in the PDF
        extract_image_block_to_payload=True # Store images as base64 data you can actually use
    )
    
    print(f"✅ Extracted {len(elements)} elements")
    return elements

# Test with your PDF file
file_path = "/home/stark/Documents/Research_pappers/Attention_is_all_you_Need.pdf"  # Change this to your PDF path
elements = partition_document(file_path)

📄 Partitioning document: /home/stark/Documents/Research_pappers/Attention_is_all_you_Need.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 3456.10it/s]


✅ Extracted 215 elements


In [4]:
def create_chunks_by_title(elements):
    """Create intelligent chunks using title-based strategy"""
    print("🔨 Creating smart chunks...")
    
    chunks = chunk_by_title(
        elements, # The parsed PDF elements from previous step
        max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
        new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
        combine_text_under_n_chars=500 # Merge tiny chunks under 500 chars with neighbors
    )
    
    print(f"✅ Created {len(chunks)} chunks")
    return chunks

# Create chunks
chunks = create_chunks_by_title(elements)

🔨 Creating smart chunks...
✅ Created 25 chunks


In [17]:
chunks[1].metadata.orig_elements[0].to_dict()["metadata"]["coordinates"]["points"]

((np.float64(745.8543090820312), np.float64(926.2586588888889)),
 (np.float64(745.8543090820312), np.float64(957.8560791015625)),
 (np.float64(963.4653082777776), np.float64(957.8560791015625)),
 (np.float64(963.4653082777776), np.float64(926.2586588888889)))

In [98]:
chunks[0].metadata.orig_elements[1].to_dict()

{'type': 'UncategorizedText',
 'element_id': '6a6d97df29c2a0c1bcf9d8b9b6d4281d',
 'text': '2023',
 'metadata': {'coordinates': {'points': ((np.float64(52.0), np.float64(599.0)),
    (np.float64(52.0), np.float64(704.0)),
    (np.float64(89.0), np.float64(704.0)),
    (np.float64(89.0), np.float64(599.0))),
   'system': 'PixelSpace',
   'layout_width': 1700,
   'layout_height': 2200},
  'last_modified': '2026-07-03T23:08:17',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': '/home/stark/Documents/Research_pappers',
  'filename': 'Attention_is_all_you_Need.pdf'}}

In [5]:
def separate_content_types(chunk):
    """Analyze what types of content are in a chunk"""
    content_data = {
        'text': chunk.text,
        'tables': [],
        'images': [],
        'types': ['text']
    }
    
    # Check for tables and images in original elements
    if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
        for element in chunk.metadata.orig_elements:
            element_type = type(element).__name__
            
            # Handle tables
            if element_type == 'Table':
                content_data['types'].append('table')
                table_html = getattr(element.metadata, 'text_as_html', element.text)
                content_data['tables'].append(table_html)
            
            # Handle images
            elif element_type == 'Image':
                if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
                    content_data['types'].append('image')
                    content_data['images'].append(element.metadata.image_base64)
    
    content_data['types'] = list(set(content_data['types']))
    return content_data

In [6]:

def create_ai_enhanced_summary(text, tables, images):
    try:
        prompt_text = f"""You are creating a searchable description of document content for retrieval.

TEXT CONTENT:
{text}

"""
        if tables:
            prompt_text += "TABLES:\n"
            for i, table in enumerate(tables):
                prompt_text += f"Table {i+1}:\n{table}\n\n"

        prompt_text += """
YOUR TASK:
Write a search index entry for this content. Plain text, no markdown.

OVERVIEW: One sentence on what this content is about.
VISUAL: Main components and labels visible in the images, in order.
TEXT: Key terms and definitions from the text or tables.
QUESTIONS: 8 short questions a user might type when looking for this content. Mix keyword-style queries ("multi-head attention diagram") and natural questions ("how does attention compute weights"). Stay on topics related to this content.
KEYWORDS: 10 search terms and synonyms.

OUTPUT:"""

        message = {"role": "user", "content": prompt_text}
        if images:
            message["images"] = images

        response = ollama.chat(
            model="qwen3-vl:4b",
            messages=[message],
            think=False,
            options={"num_ctx": 8192, "temperature": 0.2},
        )
        return response["message"]["content"]

    except Exception as e:
        print(f"❌ AI summary failed: {e}")
        return f"{text[:300]}..."

In [ ]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    documents = []
    meta_data = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']

        documents.append(enhanced_content)
        meta_data.append({"page_no" : chunk.metadata.orig_elements})

In [90]:
def get_chunk_locations(chunks):
    locations = {}
    for i, chunk in enumerate(chunks):
        per_page = {}   # page -> {"xs": [], "ys": [], "layout": (w, h)}
        for el in chunk.metadata.orig_elements:
            md = el.to_dict()["metadata"]
            pg = md.get("page_number")
            c = md.get("coordinates")
            if pg is None or not c:
                continue
            d = per_page.setdefault(pg, {"xs": [], "ys": [], "layout": None})
            for x, y in c["points"]:
                d["xs"].append(float(x))
                d["ys"].append(float(y))
            d["layout"] = (c["layout_width"], c["layout_height"])

        locations[i] = {
            "pages": sorted(per_page),
            "boxes": {
                pg: {
                    "bbox": [min(d["xs"]), min(d["ys"]), max(d["xs"]), max(d["ys"])],
                    "layout": list(d["layout"]),
                }
                for pg, d in per_page.items()
            },
        }
    return locations


In [94]:

locs = get_chunk_locations(chunks)
print(locs[24])

{'pages': [13, 14, 15], 'boxes': {13: {'bbox': [298.36395263671875, 201.40676888888876, 1407.3826904296875, 2088.999782222222], 'layout': [1700, 2200]}, 14: {'bbox': [294.85382080078125, 418.08734130859375, 1408.83349609375, 2088.999782222222], 'layout': [1700, 2200]}, 15: {'bbox': [300.0, 511.7849426269531, 1401.84765625, 2088.999782222222], 'layout': [1700, 2200]}}}


In [69]:
# page_numbers__ = get_page_numbers(chunks)
# page_numbers__

In [ ]:
import chromadb
chroma_client = chromadb.Client()

In [ ]:
collection = chroma_client.create_collection(name="my_collection")

In [ ]:
collection.add(
    ids=["id1", "id2"],
    documents=[
        "This is a document about pineapple",
        "This is a document about oranges"
    ],
    metadatas=["page1", "page2"]
)

In [ ]:
def summarise_chunks(chunks):
    """Process all chunks with AI Summaries"""
    print("🧠 Processing chunks with AI Summaries...")
    
    documents = []
    total_chunks = len(chunks)
    
    for i, chunk in enumerate(chunks):
        current_chunk = i + 1
        print(f"   Processing chunk {current_chunk}/{total_chunks}")
        
        # Analyze chunk content
        content_data = separate_content_types(chunk)
        
        # Debug prints
        print(f"     Types found: {content_data['types']}")
        print(f"     Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")
        
        # Create AI-enhanced summary if chunk has tables/images
        if content_data['tables'] or content_data['images']:
            print(f"     → Creating AI summary for mixed content...")
            try:
                enhanced_content = create_ai_enhanced_summary(
                    content_data['text'],
                    content_data['tables'], 
                    content_data['images']
                )
                print(f"     → AI summary created successfully")
                print(f"     → Enhanced content preview: {enhanced_content[:200]}...")
            except Exception as e:
                print(f"     ❌ AI summary failed: {e}")
                enhanced_content = content_data['text']
        else:
            print(f"     → Using raw text (no tables/images)")
            enhanced_content = content_data['text']
        
        # Create LangChain Document with rich metadata
        doc = Document(
            page_content=enhanced_content,
            metadata={
                "original_content": json.dumps({
                    "raw_text": content_data['text'],
                    "tables_html": content_data['tables'],
                    "images_base64": content_data['images']
                })
            }
        )
        
        documents.append(doc)
    
    print(f"✅ Processed {len(documents)} chunks")
    return documents